# Dataset setup and inspection

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data" / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project folder.")
DATA_PATH = ROOT / "data" / "raw" / "PS_20174392719_1491204439457_log.csv"
print(f"Project folder: {ROOT}")
print(f"Dataset: {DATA_PATH}")

Project folder: C:\Users\amank\ML Project\Financial-fraud-detection
Dataset: C:\Users\amank\ML Project\Financial-fraud-detection\data\raw\PS_20174392719_1491204439457_log.csv


## 1. Confirm the dataset is present

Follow the instructions in `data/README.md` if the CSV is missing.


In [2]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. See data/README.md.")
print(f"Dataset available: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB")


Dataset available: 493.5 MB


## 2. Inspect the columns and a few rows


In [3]:
sample = pd.read_csv(DATA_PATH, nrows=5)
expected = {"step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig", "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"}
missing = expected - set(sample.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")
print("Columns:", list(sample.columns))
sample


Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 3. Count fraud and legitimate transactions

Only the target column is loaded in chunks to limit memory use.


In [4]:
counts = pd.Series(dtype="int64")
for chunk in pd.read_csv(DATA_PATH, usecols=["isFraud"], chunksize=500_000):
    counts = counts.add(chunk["isFraud"].value_counts(), fill_value=0)
counts = counts.astype("int64").sort_index()
summary = pd.DataFrame({"transactions": counts, "percentage": counts / counts.sum() * 100})
summary.index.name = "isFraud (0 = legitimate, 1 = fraud)"
print(f"Total transactions: {counts.sum():,}")
summary


Total transactions: 6,362,620


,transactions,percentage
"isFraud (0 = legitimate, 1 = fraud)",,
0,6354407,99.870918
1,8213,0.129082
